# Week 03 Practice: Spark Structured Streaming + Kafka

Four progressive parts:

1. **Kafka basics** — start the broker, create topics, produce and consume messages, inspect partition ordering and consumer-group lag
2. **Structured Streaming end-to-end** — `readStream.format("kafka")`, parse JSON, write to a sink, produce messages while the query is running
3. **Windowed aggregation** — tumbling windows on event time, late-data behaviour, watermarks, output modes
4. **Project patterns** — writing to a file sink, checkpoint restart correctness, streaming-static join

Run all cells **in order**. Each section builds on the previous one.

---
## Setup

In [ ]:
# Install the Kafka client library used for producing / consuming in Python cells
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kafka-python-ng"])
print("kafka-python-ng ready")

In [ ]:
import os

# The Kafka connector is loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# If you are running outside Docker (not recommended), set the variable manually here.
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0 pyspark-shell",
)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("bdm_week03")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# If the version here differs from the version in compose.yml PYSPARK_SUBMIT_ARGS,
# update that line and restart the stack (docker compose down && docker compose up -d).
print(f"Spark {spark.version}")

In [ ]:
# Connection string for the Kafka broker on the internal Docker Compose network.
# The hostname 'kafka' is resolved by Docker DNS to the broker container.
BOOTSTRAP = "kafka:9092"
TOPIC     = "sensor-events"

---
## Part 1 — Kafka Infrastructure

### 1.1 Verify the stack is running

In a terminal (outside Jupyter), run:

```bash
docker ps
```

Both `kafka` and `week03_jupyter` should show **Up**.

### 1.2 CLI reference — for reference only, run on a host terminal

> **Note:** The code cells below use Python (`kafka-python-ng`) to do the same things.
> You do not need to run any of the commands in this section to complete the notebook.
> They are shown so you can see what the Python API maps to at the CLI level.
>
> Run these from a **host machine terminal** (not from inside Jupyter — `docker` is not
> available inside the container).

```bash
# Create a topic with 3 partitions
docker exec kafka sh -c "/opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create --topic sensor-events --partitions 3 --replication-factor 1"

# List all topics
docker exec kafka sh -c "/opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --list"

# Describe the topic — shows partitions, leader, replicas
docker exec kafka sh -c "/opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 --describe --topic sensor-events"

# Produce free-text messages (Ctrl-C to exit)
docker exec -it kafka sh -c "/opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 --topic sensor-events"

# Consume all messages from the beginning
docker exec kafka sh -c "/opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic sensor-events --from-beginning"

# Produce with explicit key  (key:value format)
docker exec -it kafka sh -c "/opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 --topic sensor-events \
  --property parse.key=true --property key.separator=:"
# then type: sensor-1:{"temperature": 22.5}

# Consume a single partition
docker exec kafka sh -c "/opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic sensor-events --partition 0 --from-beginning"
```

**Key observation:** produce several messages with the same key, then consume each
partition separately. You will see that all messages for a given key land on the same
partition — ordering is guaranteed *within* a partition but not *across* partitions.

In [ ]:
# Create the topic programmatically (same result as the CLI command above)
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP, client_id="week03-admin")

try:
    admin.create_topics([
        NewTopic(name=TOPIC, num_partitions=3, replication_factor=1)
    ])
    print(f"Topic '{TOPIC}' created with 3 partitions")
except TopicAlreadyExistsError:
    print(f"Topic '{TOPIC}' already exists")

print("Topics:", admin.list_topics())
admin.close()

In [ ]:
# Produce 9 keyed messages — 3 sensors, 3 messages each.
# Kafka's default partitioner hashes the key, so the same key always goes
# to the same partition → ordering guaranteed within that partition.

import json, random
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode(),
    value_serializer=lambda v: json.dumps(v).encode(),
)

sensors = ["sensor-1", "sensor-2", "sensor-3"]
print(f"{'Partition':>10} {'Offset':>8} {'Key':>12}  Value")
print("-" * 60)
for i in range(9):
    sensor = sensors[i % 3]
    msg = {"sensor_id": sensor, "temperature": round(20 + random.uniform(-2, 2), 2)}
    future = producer.send(TOPIC, key=sensor, value=msg)
    meta   = future.get(timeout=10)
    print(f"{meta.partition:>10} {meta.offset:>8} {sensor:>12}  {msg}")

producer.close()
print("\nNotice: each sensor always lands on the same partition.")

In [ ]:
# Consume all messages and display their partition / offset
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    auto_offset_reset="earliest",
    group_id="week03-inspect-group",
    consumer_timeout_ms=4000,       # stop after 4 s of no new messages
    value_deserializer=lambda v: json.loads(v.decode()),
    key_deserializer=lambda k: k.decode() if k else None,
)

print(f"{'Partition':>10} {'Offset':>8} {'Key':>12}  Value")
print("-" * 60)
for msg in consumer:
    print(f"{msg.partition:>10} {msg.offset:>8} {str(msg.key):>12}  {msg.value}")

consumer.close()

In [ ]:
# Inspect consumer-group offsets — equivalent to:
# docker exec kafka sh -c "/opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 \
#      --describe --group week03-inspect-group"

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
offsets = admin.list_consumer_group_offsets("week03-inspect-group")

print(f"{'TopicPartition':40} {'CommittedOffset':>15}")
print("-" * 57)
for tp, meta in sorted(offsets.items()):
    print(f"{str(tp):40} {meta.offset:>15}")

admin.close()
print("\nLag = latest_offset − committed_offset.")
print("Lag = 0 means the consumer is fully caught up.")

---
## Part 2 — Structured Streaming: Minimal End-to-End

Goal: go from zero to a working `readStream / writeStream` in one sitting.

The raw Kafka DataFrame always has this schema regardless of your topic's payload:

| Column | Type | Notes |
|--------|------|-------|
| `key` | binary | message key |
| `value` | binary | message payload — this is what you parse |
| `topic` | string | topic name |
| `partition` | int | partition number |
| `offset` | long | per-partition offset |
| `timestamp` | timestamp | Kafka broker timestamp |
| `timestampType` | int | 0 = CreateTime, 1 = LogAppendTime |

In [ ]:
# Read from Kafka. The DataFrame is lazy — nothing moves yet.
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")  # include messages already in the topic
    .load()
)

print("isStreaming:", raw_df.isStreaming)   # True — not a batch DataFrame
print()
raw_df.printSchema()

In [ ]:
# Parse the value column from binary → JSON → individual fields.
# The schema must be declared explicitly; inference is not available for streams.
# Part 1 messages didn't include event_time, so that field will be null for them.

event_schema = "sensor_id STRING, temperature DOUBLE, event_time STRING"

parsed_df = (
    raw_df
    .select(
        F.col("key").cast("string").alias("key"),
        F.from_json(F.col("value").cast("string"), event_schema).alias("d"),
        "partition",
        "offset",
        F.col("timestamp").alias("kafka_time"),
    )
    .select("key", "d.*", "partition", "offset", "kafka_time")
)

print("Parsed schema:")
parsed_df.printSchema()

In [ ]:
# Write the stream to the 'memory' sink so we can query it with spark.sql().
# The memory sink is for development / debugging only — do not use in production.

import time

for q in spark.streams.active:
    if q.name == "raw_events":
        q.stop()

raw_query = (
    parsed_df.writeStream
    .format("memory")
    .queryName("raw_events")          # table name for spark.sql()
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-raw")
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(8)   # wait for two trigger cycles
print("Messages received so far:")
spark.sql("""
    SELECT key, sensor_id, temperature, event_time, partition, offset, kafka_time
    FROM raw_events
    ORDER BY partition, offset
""").show(truncate=False)

In [ ]:
# Produce a few more messages while the query is still running.
# After the next trigger fires, they will appear in the table.

from datetime import datetime, timezone

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode(),
    value_serializer=lambda v: json.dumps(v).encode(),
)
for sensor in ["sensor-1", "sensor-2", "sensor-3"]:
    msg = {
        "sensor_id": sensor,
        "temperature": round(21 + random.uniform(0, 5), 2),
        "event_time": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
    }
    producer.send(TOPIC, key=sensor, value=msg)
producer.flush()
producer.close()

time.sleep(8)   # wait for the next trigger
print("After producing more messages:")
spark.sql("""
    SELECT key, sensor_id, temperature, event_time, partition, offset
    FROM raw_events
    ORDER BY partition, offset
""").show(truncate=False)

In [ ]:
raw_query.stop()
print("Part 2 query stopped.")

---
## Part 3 — Windowed Aggregation with Event Time

### Key concepts

| Term | Meaning |
|------|---------|
| **Event time** | Timestamp *embedded in the message payload*, set by the producer |
| **Processing time** | Wall-clock time when Spark processes the message |
| **Tumbling window** | Fixed-size, non-overlapping windows; each event belongs to exactly one window |
| **Watermark** | Tells Spark how far behind event time can lag; older windows are dropped |
| **Output mode** | `append` — new rows only; `complete` — full result table every trigger; `update` — only changed rows |

Without a watermark, Spark keeps every window in state indefinitely (correct but memory-intensive).
With a watermark of *T*, Spark drops any window whose end time is older than `max(event_time_seen) − T`.

In [ ]:
# Re-read the stream from 'latest' so Part 3 only sees messages we produce below.
# event_time is now parsed as TIMESTAMP so Spark can use it for windowing.

sensor_schema = "sensor_id STRING, temperature DOUBLE, event_time STRING"

stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")   # only new messages
    .load()
    .select(
        F.from_json(F.col("value").cast("string"), sensor_schema).alias("d")
    )
    .select(
        "d.sensor_id",
        "d.temperature",
        F.to_timestamp("d.event_time", "yyyy-MM-dd'T'HH:mm:ss").alias("event_time"),
    )
    .filter(F.col("event_time").isNotNull())   # drop Part 1 messages (no event_time)
)

stream_df.printSchema()

In [ ]:
# Helper: produce events with a controllable event_time offset.
# time_offset_minutes < 0  → event time in the past (late events)
# time_offset_minutes = 0  → event time is 'now'

from datetime import datetime, timezone, timedelta

def produce_events(n=5, time_offset_minutes=0, temperature_base=22.0):
    p = KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        key_serializer=lambda k: k.encode(),
        value_serializer=lambda v: json.dumps(v).encode(),
    )
    ts = (datetime.now(timezone.utc) + timedelta(minutes=time_offset_minutes)).strftime(
        "%Y-%m-%dT%H:%M:%S"
    )
    for i in range(n):
        sensor = f"sensor-{(i % 3) + 1}"
        msg = {
            "sensor_id": sensor,
            "temperature": round(temperature_base + random.uniform(-1, 1), 2),
            "event_time": ts,
        }
        p.send(TOPIC, key=sensor, value=msg)
    p.flush()
    p.close()
    print(f"Produced {n} events, event_time offset = {time_offset_minutes:+d} min  ({ts})")

### 3.1 Tumbling window — complete mode, no watermark

Without a watermark, Spark keeps every window in state forever.
If a late event arrives for an old window, that window is recomputed and re-emitted.
You will see this happen in the cells below.

In [ ]:
for q in spark.streams.active:
    if q.name == "windowed_complete":
        q.stop()

windowed_df = (
    stream_df
    .groupBy(
        F.window("event_time", "1 minute").alias("window"),
        "sensor_id",
    )
    .agg(
        F.round(F.avg("temperature"), 2).alias("avg_temp"),
        F.count("*").alias("n"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "sensor_id", "avg_temp", "n",
    )
)

windowed_query = (
    windowed_df.writeStream
    .format("memory")
    .queryName("windowed_complete")
    .outputMode("complete")            # full table re-emitted every trigger
    .option("checkpointLocation", "/tmp/chk-windowed-complete")
    .trigger(processingTime="5 seconds")
    .start()
)

print("Windowed query started (complete mode, no watermark).")

In [ ]:
# Produce normal events (event_time = now) and inspect the window state.

produce_events(n=6, time_offset_minutes=0, temperature_base=22.0)
time.sleep(10)

spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_complete
    ORDER BY window_start, sensor_id
""").show(truncate=False)

In [ ]:
# Produce late events — 10 minutes in the past.
# Without a watermark, Spark has no way to discard old windows.
# The window from 10 minutes ago will be created and included in the complete result.
# (temperature_base=99.9 makes them easy to spot)

produce_events(n=3, time_offset_minutes=-10, temperature_base=99.9)
time.sleep(10)

print("After late events — old window appears (no watermark = Spark keeps all state):")
spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_complete
    ORDER BY window_start, sensor_id
""").show(truncate=False)

In [ ]:
windowed_query.stop()
print("Complete-mode query stopped.")

### 3.2 Tumbling window — update mode + watermark

A **watermark** of 5 minutes tells Spark:
> "Once the maximum event time seen exceeds a window's end by more than 5 minutes,
> that window will never receive data again — drop its state."

Late events that fall outside the watermark boundary are silently dropped.

**`update` output mode** emits only the rows that changed in each trigger
(vs `complete` which re-emits the entire table). For windowed aggregations with a
watermark, `update` is typically more efficient.

In [ ]:
for q in spark.streams.active:
    if q.name == "windowed_update":
        q.stop()

watermarked_df = (
    stream_df
    .withWatermark("event_time", "5 minutes")   # discard state for windows older than this
    .groupBy(
        F.window("event_time", "1 minute").alias("window"),
        "sensor_id",
    )
    .agg(
        F.round(F.avg("temperature"), 2).alias("avg_temp"),
        F.count("*").alias("n"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "sensor_id", "avg_temp", "n",
    )
)

watermarked_query = (
    watermarked_df.writeStream
    .format("memory")
    .queryName("windowed_update")
    .outputMode("update")              # only changed rows per trigger
    .option("checkpointLocation", "/tmp/chk-windowed-update")
    .trigger(processingTime="5 seconds")
    .start()
)

print("Windowed query started (update mode, watermark = 5 min).")

In [ ]:
# Produce normal events and observe the current window.

produce_events(n=6, time_offset_minutes=0, temperature_base=22.0)
time.sleep(10)

print("Current windows (update mode, watermark = 5 min):")
spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_update
    ORDER BY window_start, sensor_id
""").show(truncate=False)

In [ ]:
# Produce very late events — 15 minutes in the past.
# Watermark = 5 min, so events 15 min old are beyond the boundary.
# Spark drops them silently; the old window does NOT appear in the result.

produce_events(n=3, time_offset_minutes=-15, temperature_base=99.9)
time.sleep(10)

print("After very late events — they are dropped (watermark = 5 min < 15 min lag):")
spark.sql("""
    SELECT window_start, window_end, sensor_id, avg_temp, n
    FROM windowed_update
    ORDER BY window_start, sensor_id
""").show(truncate=False)

print("No row with avg_temp ≈ 99.9 → late events were discarded.")

In [ ]:
# Stop all running streams before Part 4
for q in spark.streams.active:
    q.stop()
print("All streams stopped.")

---
## Part 4 — Project Patterns

The two patterns below are the foundation of every medallion-architecture streaming pipeline
(like the one in Project 2). Working through them here means you will have seen the building
blocks before you encounter them in the project.

### 4.1 Writing to a file sink and verifying checkpoint restart correctness

In production (and in Project 2) you write to a table (Iceberg, Delta, …) rather than to
memory. A **file-based sink** works the same way. The critical guarantee is:

> After a restart, the query reads from the **committed offsets** stored in the checkpoint
> directory — not from `startingOffsets`. Messages that were already written are never
> re-processed, so the output contains no duplicates.

You will see this below by comparing row counts before and after a restart.

In [ ]:
import shutil

# Clean up any leftover files from a previous run of this cell
shutil.rmtree("/tmp/bronze",     ignore_errors=True)
shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)

# Build the parsed stream — same logic as Part 2 but reading from 'earliest'
# so we ingest everything already in the topic (simulates a fresh pipeline start).
bronze_source = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("value").cast("string").alias("raw_value"),  # keep payload as-is
        "partition",
        "offset",
        F.col("timestamp").alias("kafka_time"),
    )
)

# Write raw events to JSON files — this is the bronze-layer pattern.
# checkpointLocation tracks which offsets have been committed.
bronze_query = (
    bronze_source.writeStream
    .format("json")
    .outputMode("append")
    .option("path",              "/tmp/bronze")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .start()
)

# Produce a few more events while the query is running
produce_events(n=4, time_offset_minutes=0)

time.sleep(15)   # let at least two triggers fire

count_before = spark.read.json("/tmp/bronze").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Query stopped.")

In [ ]:
# Restart the query using the SAME checkpoint directory.
# Even though startingOffsets is still 'earliest', Spark ignores that setting
# and resumes from the committed offsets in the checkpoint.

bronze_query2 = (
    bronze_source.writeStream
    .format("json")
    .outputMode("append")
    .option("path",              "/tmp/bronze")
    .option("checkpointLocation", "/tmp/chk-bronze")  # same checkpoint!
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(12)   # wait for two triggers

count_after = spark.read.json("/tmp/bronze").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()
if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — "
          "those are new messages produced between the two runs.")

bronze_query2.stop()

### 4.2 Streaming-static join (enrichment / silver-layer pattern)

A **streaming-static join** combines a live stream with a DataFrame that was loaded
once from a file, database, or inline data. The static side is broadcast to all
Spark executors and held in memory for the lifetime of the query.

This is how you would enrich events with a **zone lookup table** (like the taxi-zone
CSV in Project 2) without the overhead of a full streaming-streaming join.

Key properties:
- The static DataFrame is read **once** at query start. Changes to the source after
  that point are invisible to the running query.
- Unmatched stream rows are kept or dropped depending on the join type (`left`, `inner`, …).
- No watermark is required (the static side has no timestamps).

In [ ]:
# Static lookup table — loaded once, broadcast to all workers.
# In the project this would be read from a CSV or Parquet file.
sensor_lookup = spark.createDataFrame(
    [
        ("sensor-1", "Building A",  "indoor"),
        ("sensor-2", "Building B",  "indoor"),
        ("sensor-3", "Rooftop",     "outdoor"),
    ],
    schema="sensor_id STRING, location STRING, sensor_type STRING",
)

sensor_lookup.show()

In [ ]:
for q in spark.streams.active:
    if q.name == "enriched_events":
        q.stop()

# Join the stream (left side) with the static lookup (right side).
# Left join: stream rows without a matching sensor_id get null for location/sensor_type.
enriched_df = (
    stream_df
    .join(F.broadcast(sensor_lookup), on="sensor_id", how="left")
)

enriched_query = (
    enriched_df.writeStream
    .format("memory")
    .queryName("enriched_events")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-enriched")
    .trigger(processingTime="5 seconds")
    .start()
)

# Produce events with sensor IDs — some will match the lookup, some won't
produce_events(n=6, time_offset_minutes=0)

time.sleep(12)

print("Enriched stream — sensor_id joined with location and sensor_type:")
spark.sql("""
    SELECT sensor_id, temperature, event_time, location, sensor_type
    FROM enriched_events
    ORDER BY event_time, sensor_id
""").show(truncate=False)

enriched_query.stop()

In [ ]:
# Stop everything before the exercises
for q in spark.streams.active:
    q.stop()
print("All streams stopped.")

---
## Exercises

You are free to use any resource (documentation, AI tools, etc.).

---
### Exercise 1 — 2-minute sliding window

Write a streaming query that:

1. Reads from the `sensor-events` topic (use `startingOffsets="latest"`).
2. Applies a **sliding window** of size **2 minutes** and slide interval **30 seconds**
   on `event_time`.
3. Aggregates by `sensor_id` and computes `min(temperature)`, `max(temperature)`,
   and `count`.
4. Adds a watermark of **3 minutes**.
5. Writes to the `memory` sink in `update` output mode.
6. Produces 10 events and shows the result table.
7. Produces 5 late events (5 minutes in the past — within watermark) and shows the
   table again. Are any windows updated?
8. Stops the stream.

**Hint:** `F.window(time_col, windowDuration, slideDuration)` takes a third argument
for the slide interval.

In [ ]:
# Exercise 1 — your answer


---
### Exercise 2 — Output mode comparison

Run the same **tumbling 1-minute window** query twice — once with `complete` output
mode and once with `update` output mode (both with a 5-minute watermark).

1. Produce 6 events in the current minute.
2. Show the memory table for each query.
3. Produce 3 more events **in the same minute window**.
4. Show the memory tables again.

Answer in a markdown cell:
- How many rows does each table have after step 4?
- Which mode re-emits unchanged windows?
- When would you prefer `update` over `complete` in production?

In [ ]:
# Exercise 2 — your answer


---
## Further reading

- [Structured Streaming + Kafka Integration Guide](https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html)
- [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) — especially the sections on event time, watermarks, and output modes
- [PySpark Streaming API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/index.html)
- [Kafka documentation](https://kafka.apache.org/documentation/)